In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
CHEMIN = "data/instagram_users_lifestyleSafe.csv"

Matplotlib is building the font cache; this may take a moment.


In [3]:
df = pd.read_csv(CHEMIN)
print(df.shape)

(200000, 58)


## Choix des colonnes

**Filtre 1 – Le droit.** Nous ne gardons que les colonnes qu'Instagram connaît pour tous ses utilisateurs et qu'il peut légalement utiliser pour du ciblage. Sont exclues : les données de santé et de bien-être (article 9 du RGPD), les loisirs et la situation sociale (déclarés dans l'enquête du panel uniquement), et les identifiants (`user_id`, `app_name`). Le stress, le bonheur et le sommeil serviront uniquement à décrire les segments.

In [5]:
CANDIDATES = [
    "daily_active_minutes_instagram", "sessions_per_day", "average_session_length_minutes",
    "posts_created_per_week", "reels_watched_per_day", "stories_viewed_per_day",
    "likes_given_per_day", "comments_written_per_day", "dms_sent_per_week", "dms_received_per_week",
    "ads_viewed_per_day", "ads_clicked_per_day",
    "time_on_feed_per_day", "time_on_explore_per_day", "time_on_messages_per_day", "time_on_reels_per_day",
    "followers_count", "following_count", "notification_response_rate", "linked_accounts_count",
    "user_engagement_score", "account_creation_year",
]
print(len(CANDIDATES), "colonnes candidates après le filtre 1")

22 colonnes candidates après le filtre 1


In [6]:
pd.set_option("display.max_columns", None)

filtre2 = pd.DataFrame({
    "valeurs_differentes": df[CANDIDATES].nunique(),
    "corr_temps_passe":    df[CANDIDATES].corrwith(df["daily_active_minutes_instagram"]).round(2),
    "corr_stress":         df[CANDIDATES].corrwith(df["perceived_stress_score"]).round(2),
    "corr_bonheur":        df[CANDIDATES].corrwith(df["self_reported_happiness"]).round(2),
}).sort_values("corr_temps_passe", ascending=False)

filtre2

,valeurs_differentes,corr_temps_passe,corr_stress,corr_bonheur
daily_active_minutes_instagram,532,1.00,0.83,-0.37
likes_given_per_day,306,0.98,0.82,-0.37
time_on_feed_per_day,305,0.97,0.81,-0.36
stories_viewed_per_day,138,0.95,0.81,-0.35
comments_written_per_day,81,0.94,0.79,-0.35
time_on_reels_per_day,203,0.93,0.78,-0.35
dms_received_per_week,89,0.92,0.77,-0.35
dms_sent_per_week,81,0.90,0.75,-0.34
time_on_messages_per_day,124,0.90,0.75,-0.33
ads_viewed_per_day,51,0.89,0.75,-0.33


**Filtre 2 – L'utilité statistique.** Aucune colonne candidate n'est constante (toutes ont au moins 6 valeurs différentes). Le tableau révèle quatre familles :

- **Copies du temps passé (corrélation ≥ 0,87)** : likes, temps sur le fil, stories, commentaires, temps sur les Reels, messages reçus et envoyés, temps en messagerie, publicités vues, temps sur Explorer, Reels regardés. Ces colonnes répètent la même information que le temps passé. Nous gardons `daily_active_minutes_instagram` pour représenter la quantité d'usage. Les autres serviront à construire des ratios qui décrivent le style d'usage.
- **Colonnes en partie liées (0,50 à 0,75)** : sessions par jour, publicités cliquées, publications par semaine. Elles contiennent une part d'information nouvelle et sont conservées pour les ratios.
- **Colonnes indépendantes (corrélation < 0,20)** : durée moyenne d'une session, abonnés, abonnements, année de création du compte, taux de réponse aux notifications, comptes liés. Elles décrivent autre chose que la quantité d'usage, mais aucune n'est liée au stress (corrélation < 0,20).
- **Colonne suspecte** : `user_engagement_score` baisse quand le temps passé augmente (corrélation de −0,53). Ce comportement est contraire à ce qu'on attend d'un score d'engagement, et son mode de calcul n'est pas documenté. Elle est exclue.

In [8]:
temps_total = df[["time_on_feed_per_day", "time_on_explore_per_day",
                  "time_on_messages_per_day", "time_on_reels_per_day"]].sum(axis=1)

df["part_reels"]         = df["time_on_reels_per_day"] / temps_total
df["part_messages"]      = df["time_on_messages_per_day"] / temps_total
df["part_explore"]       = df["time_on_explore_per_day"] / temps_total
df["ratio_createur"]     = (df["posts_created_per_week"] / 7 + df["comments_written_per_day"]) / (df["likes_given_per_day"] + df["stories_viewed_per_day"] + 1)
df["taux_clic_pub"]      = df["ads_clicked_per_day"] / (df["ads_viewed_per_day"] + 1)
df["ratio_abonnes"]      = df["followers_count"] / (df["following_count"] + 1)
df["sessions_par_heure"] = df["sessions_per_day"] / (df["daily_active_minutes_instagram"] / 60)

RATIOS = ["part_reels", "part_messages", "part_explore", "ratio_createur",
          "taux_clic_pub", "ratio_abonnes", "sessions_par_heure"]
print(len(RATIOS), "ratios créés")

7 ratios créés


In [9]:
filtre3 = pd.DataFrame({
    "moyenne":          df[RATIOS].mean().round(3),
    "coef_variation":   (df[RATIOS].std() / df[RATIOS].mean()).round(2),
    "corr_temps_passe": df[RATIOS].corrwith(df["daily_active_minutes_instagram"]).round(2),
    "corr_stress":      df[RATIOS].corrwith(df["perceived_stress_score"]).round(2),
    "corr_bonheur":     df[RATIOS].corrwith(df["self_reported_happiness"]).round(2),
})
filtre3

,moyenne,coef_variation,corr_temps_passe,corr_stress,corr_bonheur
part_reels,0.255,0.17,0.02,0.01,-0.01
part_messages,0.150,0.23,-0.05,-0.04,0.03
part_explore,0.169,0.25,-0.00,-0.00,0.00
ratio_createur,0.154,0.22,0.36,0.29,-0.14
taux_clic_pub,0.234,0.62,0.06,0.05,-0.02
ratio_abonnes,0.901,0.57,0.01,0.01,-0.01
sessions_par_heure,3.794,0.64,-0.35,-0.28,0.19


In [10]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

for cible in ["perceived_stress_score", "self_reported_happiness"]:
    y = df[cible]
    r2_temps   = cross_val_score(LinearRegression(), df[["daily_active_minutes_instagram"]], y, cv=5, scoring="r2").mean()
    r2_complet = cross_val_score(LinearRegression(), df[["daily_active_minutes_instagram"] + RATIOS], y, cv=5, scoring="r2").mean()
    print(f"{cible} : temps seul = {r2_temps:.3f} | temps + ratios = {r2_complet:.3f}")

perceived_stress_score : temps seul = 0.696 | temps + ratios = 0.696
self_reported_happiness : temps seul = 0.140 | temps + ratios = 0.143


**Filtre 3 – La pertinence.** Nous avons créé 7 ratios qui décrivent la façon d'utiliser Instagram plutôt que la quantité d'usage.

- **Ils apportent une information nouvelle sur l'usage.** Cinq d'entre eux (part des Reels, des messages, d'Explorer, taux de clic publicitaire, ratio abonnés/abonnements) ont une corrélation inférieure à 0,06 avec le temps passé, et un coefficient de variation supérieur à 0,15 : les utilisateurs se différencient réellement sur ces critères.
- **Ils n'apportent rien sur le bien-être.** Le temps passé seul explique 69,6 % des différences de stress déclaré (R², validation croisée en 5 plis). L'ajout des 7 ratios ne change pas ce score (69,6 %). Pour le bonheur, le gain est négligeable (de 14,0 % à 14,3 %).

**Conclusion :** chez les volontaires du panel, le bien-être déclaré est lié à la quantité d'usage, pas à la façon d'utiliser l'application. Les ratios restent utiles pour décrire des profils d'usage destinés aux équipes produit. Il s'agit d'une association statistique, et non d'une relation de cause à effet.